# Day 24 — Percentiles, Quartiles, and Quantiles

This notebook covers Percentiles, Quartiles ($Q_1, Q_2, Q_3$), Interquartile Range (IQR), Tukey Outlier Fences, and Tail-Latency Monitoring ($p95, p99$)

## Validation goals
- Define percentiles and compute custom positional ranks using NumPy and Pandas
- Distinguish quartiles ($Q_1, Q_2, Q_3$) and their relationship with the median
- Calculate the Interquartile Range (IQR) and implement Tukey’s outlier fence method
- Analyze tail-latency amplification in distributed microservices using $p95$ and $p99$ metrics
- Evaluate why rank-based statistics are robust to extreme outliers compared to moment-based metrics

While central tendency tells you where the middle is and dispersion tells you how wide the spread is, neither metric reveals the shape of the distribution or where specific data points sit relative to the rest of the dataset.

Percentiles solve this by dividing a ranked dataset into 100 equal parts. They are essential in exploratory data analysis, outlier detection, performance engineering (such as tracking 95th or 99th percentile server latency), and machine learning feature engineering.

### Core Concepts & Deep Dive1 

1. **Definition of a Percentile** 
The $p$-th percentile ($P_p$) is the value below which a given percentage ($p$) of the observations in a group of observations fall.
- For example, if your test score is at the 85th percentile, it means 85% of the scores are lower than or equal to yours, and 15% are higher.

2. **Special Quantiles:**
 Quartiles and the MedianQuantiles are general divisions of data into equal-sized subsets. Percentiles divide data into 100 parts, while quartiles divide data into 4 equal parts:
- $Q_1$ (First Quartile / 25th Percentile): 25% of the data falls below this point.
- $Q_2$ (Second Quartile / 50th Percentile): Exactly equal to the median.
- $Q_3$ (Third Quartile / 75th Percentile): 75% of the data falls below this point.

3. **Interquartile Range (IQR) and Outlier Detection**
The Interquartile Range (IQR) measures the statistical dispersion of the middle 50% of a dataset:$$\text{IQR} = Q_3 - Q_1$$

Unlike standard deviation, the IQR is entirely robust to extreme outliers because it ignores the top and bottom 25% of values. The IQR is the foundation of the standard Tukey Outlier Fence method used in boxplots:
- Lower Outlier Bound: $Q_1 - 1.5 \times \text{IQR}$
- Upper Outlier Bound: $Q_3 + 1.5 \times \text{IQR}$

Any data point falling outside these fences is typically flagged and removed or capped as an outlier.

4. **System Performance Metrics: p95 and p99**
In production software engineering and AI infrastructure monitoring, metrics like mean response time can mask critical failures. If 99 requests take 20ms and 1 request takes 10,000ms, the mean looks acceptable, but users experience severe lag. Tracking the 95th (p95) and 99th (p99) percentiles exposes these tail-latency spikes.

### Calculating Percentiles in Python

Here is a script demonstrating how to calculate custom percentiles, quartiles, and compute outlier bounds using NumPy and Pandas.

In [1]:
import numpy as np
import pandas as pd

# Generate a dataset of server response latencies with extreme spikes
np.random.seed(42)
latencies = np.concatenate([
    np.random.normal(loc=40.0, scale=8.0, size=950),
    np.random.uniform(200.0, 1500.0, size=50) # tail latency spikes
])

df = pd.DataFrame({"latency_ms": latencies})

# ==========================================
# 1. Computing Percentiles and Quartiles
# ==========================================
p25 = np.percentile(df["latency_ms"], 25) # Also Q1
median_val = np.percentile(df["latency_ms"], 50) # Also Q2 / Median
p75 = np.percentile(df["latency_ms"], 75) # Also Q3
p95 = np.percentile(df["latency_ms"], 95)
p99 = np.percentile(df["latency_ms"], 99)

print(f"--- Latency Percentile Summary ---")
print(f"25th Percentile (Q1): {p25:.2f} ms")
print(f"50th Percentile (Median): {median_val:.2f} ms")
print(f"75th Percentile (Q3): {p75:.2f} ms")
print(f"95th Percentile (p95): {p95:.2f} ms")
print(f"99th Percentile (p99): {p99:.2f} ms")

# ==========================================
# 2. Outlier Detection using the IQR Method
# ==========================================
iqr = p75 - p25
upper_bound = p75 + 1.5 * iqr
lower_bound = p25 - 1.5 * iqr

outliers = df[df["latency_ms"] > upper_bound]

print(f"\n--- Outlier Detection ---")
print(f"Calculated IQR: {iqr:.2f}")
print(f"Upper Outlier Threshold: {upper_bound:.2f} ms")
print(f"Number of Outliers Detected: {len(outliers)}")

--- Latency Percentile Summary ---
25th Percentile (Q1): 35.20 ms
50th Percentile (Median): 40.64 ms
75th Percentile (Q3): 46.15 ms
95th Percentile (p95): 78.04 ms
99th Percentile (p99): 1214.02 ms

--- Outlier Detection ---
Calculated IQR: 10.95
Upper Outlier Threshold: 62.57 ms
Number of Outliers Detected: 52


### Guided Question:

Suppose an API gateway reports an average (mean) response time of 45ms, but the p99 response time is 3,200ms.

Why does relying solely on the mean hide critical user experience degradation, and why are p95 and p99 metrics critical for distributed systems?

### Answer:

1. **Why the Mean Hides User Experience Degradation**
The arithmetic mean is heavily influenced by volume rather than distribution. In the code example provided, 950 out of 1,000 requests (95%) execute quickly around 40ms, while the remaining 50 requests experience extreme tail latencies up to 1,500ms.
- The Math of Masking: Because the vast majority of requests cluster tightly around the normal distribution, they pull the mathematical average down to a seemingly healthy 45ms.

- The Real User Impact: Averages do not represent individual experiences. If 5% of your users (which could translate to tens of thousands of people daily in a high-traffic application) wait over a second or seconds for a response, the system feels sluggish and broken to them, even though operations dashboards show a green "45ms average" status.

2. **Why p95 and p99 Metrics Are Critical for Distributed Systems**
High-percentile metrics (such as p95, p99, and p99.9) measure the latency experienced by the slowest 5% or 1% of requests, making them essential for modern architectures for two primary reasons:

- Tail Latency Amplification in Microservices: Modern applications rarely rely on a single monolithic database query; instead, they use microservices with complex fan-out patterns. A single user action often triggers parallel calls to 10, 20, or 50 downstream services (e.g., authentication, inventory, pricing, recommendations).If a single service has a p99 latency of 300ms, and a user request fans out to 20 independent services, the probability that the user's overall request hits at least one slow service is calculated as:$$\text{Probability of delay} = 1 - (0.99)^{20} \approx 18.2\%$$

Nearly one in five users will experience a laggy request, even though each individual microservice appears to perform well 99% of the time. Monitoring p95 and p99 exposes these compounding bottlenecks.

- Identifying Infrastructure and Resource Bottlenecks: Tail latencies are rarely random; they are symptom indicators of underlying system stress, such as:

    - Garbage collection pauses in managed runtimes (like Java or Node.js).
    - Database connection pool exhaustion or lock contention.
    - Network jitter, packet loss, or noisy neighbors in cloud environments.
    - Thread starvation under sudden traffic spikes.

By tracking p95 and p99, engineers catch resource degradation and architectural bottlenecks long before they drag down the overall system average or trigger total outages.

### Guided Question:

How do percentiles and the Interquartile Range (IQR) maintain stability when extreme outliers are introduced into a dataset, whereas the mean and standard deviation are heavily distorted?

### Answer

Percentiles and the Interquartile Range (IQR) maintain stability in the presence of extreme outliers because they are rank-based (non-parametric) statistics, whereas the mean and standard deviation are moment-based (magnitude-based) statistics.

The fundamental mathematical differences explain why extreme values distort one set of metrics while leaving the other anchored.

1. **Why the Mean and Standard Deviation Are Distorted**
Both the mean and standard deviation incorporate the exact numerical magnitude of every single data point, making them vulnerable to distortion by outliers
- The Mean ($\bar{x}$): Calculated as the sum of all values divided by the sample size ($n$):$$\bar{x} = \frac{1}{n} \sum_{i=1}^{n} x_i$$
Because every value is added directly into the numerator, a single extreme outlier (e.g., a server latency spike of $100,000\text{ ms}$) injects massive weight into the sum. This pulls the mean upward, misrepresenting the typical experience of the vast majority of users.
- The Standard Deviation ($s$): Calculated using squared deviations from the mean:$$s = \sqrt{\frac{\sum_{i=1}^{n} (x_i - \bar{x})^2}{n-1}}$$
Because the differences are squared, outliers do not just add linear weight—they are amplified quadratically. An extreme outlier creates a massive squared distance, causing the standard deviation to balloon and rendering it useless as a measure of spread for skewed distributions.

2. **Why Percentiles and IQR Remain Stable**
Percentiles and the IQR do not care how large an extreme value is; they only care about where values rank in a sorted sequence.
- Rank-Order Invariance: To compute percentiles, data is sorted in ascending order. The 50th percentile (median) or 99th percentile represents the value at a specific positional index, regardless of its magnitude. If a request latency spike at the 99th percentile goes from $1,500\text{ ms}$ to $100,000\text{ ms}$, its positional rank as the 99th percentile remains identical. The numerical value changes only for that specific index, leaving all lower percentiles completely unaffected.
- The Boundaries of IQR: The Interquartile Range is defined as:$$\text{IQR} = Q_3 - Q_1$$
$Q_1$ (25th percentile) and $Q_3$ (75th percentile) describe the bounds of the middle 50% of the data. Because extreme outliers reside far out in the upper tail (well above $Q_3$), they are completely excluded from the arithmetic used to compute the IQR. Changing the severity or magnitude of those outliers has zero impact on $Q_1$ or $Q_3$, ensuring the calculated spread and outlier thresholds remain rock-solid.

### Guided Question:

When computing percentiles on small datasets where the exact target rank falls between two discrete data points, software libraries use interpolation methods (e.g., linear interpolation).

Why might two different data science libraries or programming languages return slightly different floating-point values for the exact same percentile calculation on a small sample size?

### Answer:

When computing percentiles on small datasets where the target rank falls between discrete data points, different programming languages or data science libraries often return slightly different values. This occurs because there is no single universally accepted mathematical standard for estimating sample percentiles.

Instead, software packages rely on different algorithmic definitions and interpolation rules. The primary drivers of these discrepancies include:

1. **Different Sample Quantile Definitions (Hyndman-Fan Types)**
Statistical software packages implement various definitions for sample quantiles. A classic paper by Hyndman and Fan (1996) identified at least nine distinct common algorithms used across software like R, SAS, Excel, and Python libraries.
- NumPy's Default (method='linear'): Computes the fractional rank using the formula $i + (j - i) \times fraction$, which aligns with R's Type 7 algorithm where points are linearly interpolated between empirical cumulative distribution function steps.
- Other Defaults (Excel / Python's statistics.quantiles): Excel uses methods closer to Weibull plotting positions, while other libraries or older versions might use midpoints, nearest-neighbor matching, or step functions that do not perform linear interpolation at all.

2. **Variations in Fractional Rank Formulas**
To find the value at percentile $p$ for a sorted array of size $N$, libraries must first calculate the target index (or rank). Different libraries define this fractional index differently:
- Some use $(N - 1)p$, treating the minimum and maximum values as the 0th and 100th percentiles anchored at the exact boundaries.
- Others use $Np$ or add small continuity corrections (such as $\frac{k - 0.5}{N}$), shifting the positional mapping of the interpolation window.

3. **Magnified Impact on Small Sample Sizes**
In large datasets ($N = 1,000,000$), the distance between adjacent sorted data points is minuscule, meaning different interpolation algorithms yield virtually identical floating-point results.

In a small sample (e.g., $N = 10$), the gap between adjacent data points represents a massive percentage of the value range. Because the distance between points is large, choosing a slightly different fractional index or interpolation weight results in a noticeable numerical discrepancy in the final output.

4. **Floating-Point Precision and Rounding**
When calculating fractional indices like $9.5$ or computing weights such as $0.25$ and $0.75$, minor differences in internal floating-point arithmetic or how rounding/casting is handled can cause slight variations in the trailing decimals of the final interpolated value.